# EGM722 Assignment - Data Download

This notebook searches for and downloads Sentinel-1 SAR data for Hurricane Beryl (July 2024).

**You need a free NASA EarthData account:** https://urs.earthdata.nasa.gov

**You may also need to accept the ASF DAAC data access agreement** the first time you download.
If the download fails with a EULA error, log in at https://urs.earthdata.nasa.gov, go to **My Profile → Applications**, find Alaska Satellite Facility Data Access, and click **Accept**. Or follow the link shown in the error.

In [1]:
import earthaccess
from pathlib import Path

In [2]:
# set up the data folder
data_dir = Path('data/raw')
data_dir.mkdir(parents=True, exist_ok=True)

print(f'Data will be saved to: {data_dir.resolve()}')

Data will be saved to: D:\EGM722\EGM722-Assignment\data\raw


## 1. Log in to NASA EarthData

In [3]:
# log in to NASA EarthData
# the first time you run this it will ask for your username and password
# after that it saves them so you won't be asked again
auth = earthaccess.login(strategy='interactive', persist=True)

Enter your Earthdata Login username:  aratajczak02
Enter your Earthdata password:  ········


## 2. Search for scenes

Hurricane Beryl made landfall near Houston on 8 July 2024.
The search window covers a few days either side to catch the nearest Sentinel-1 pass.

In [4]:
# bounding box for Houston/Gulf Coast, Texas (west, south, east, north)
bbox = (-96.5, 28.8, -94.5, 30.5)

results = earthaccess.search_data(
    short_name='OPERA_L2_RTC-S1_V1',
    bounding_box=bbox,
    temporal=('2024-07-05', '2024-07-14'),
    count=50
)

print(f'Found {len(results)} scene(s)')

Found 50 scene(s)


C:\Users\arata\anaconda3\envs\egm722-assignment\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


## 3. Download scenes

In [5]:
# download to data/raw - this can take a few minutes
files = earthaccess.download(results, local_path=str(data_dir))
print(f'Downloaded {len(files)} file(s)')

C:\Users\arata\anaconda3\envs\egm722-assignment\Lib\site-packages\earthaccess\store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/200 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/200 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/200 [00:00<?, ?it/s]

Downloaded 200 file(s)


In [6]:
tif_files = list(data_dir.glob('**/*.tif'))
print(f'{len(tif_files)} GeoTIFF file(s) downloaded:')

150 GeoTIFF file(s) downloaded:


## 4. Download WorldPop population raster

WorldPop provides a gridded population count for the USA at 1 km resolution.
This is used in notebook 02 to estimate how many people lived in the flooded area.

In [ ]:
import urllib.request

# WorldPop USA 2020 population count, 1 km resolution
worldpop_url = 'https://data.worldpop.org/GIS/Population/Global_2000_2020_1km/2020/USA/usa_ppp_2020_1km_Aggregated.tif'
worldpop_path = data_dir / 'worldpop_usa_2020_1km.tif'

if worldpop_path.exists():
    print(f'WorldPop raster already downloaded: {worldpop_path}')
else:
    urllib.request.urlretrieve(worldpop_url, worldpop_path)
    print(f'Saved: {worldpop_path}')